# SPACECONS — all base methods on Kaggle (DPT · PAGA · SpaceFlow · SpaTrack)

**Before you run (right-hand panel):** Settings → **Internet: On** · Accelerator → **GPU T4** (optional) · **Add Input** → your MOSTA dataset (`E9.5_E1S1.MOSTA.h5ad`).

Then **Run All**. Every method saves `pt_<method>.csv` to `/kaggle/working`.
At the end, click **Save Version → Save & Run All (Commit)** so the files are kept permanently (a session restart wipes `/kaggle/working`).

| # | Method | Uses position? | Family |
|---|---|---|---|
| 1 | DPT | No | Diffusion (random walk) |
| 2 | PAGA | No | Cluster graph |
| 3 | SpaceFlow | Yes | Graph neural network + diffusion |
| 4 | SpaTrack | Yes | Optimal transport (expression + distance) |


In [1]:
# Install (internet must be ON). The pip 'dependency conflict' messages are expected and harmless.
# !pip install -q --no-deps SpaceFlow spaTrack
# !pip install -q torch-geometric gudhi cmcrameri scanpy igraph leidenalg pot pygam

## 0. Load data, select the lineage, preprocess, and fix the root (same as PyCharm)

In [3]:
import glob, os
import numpy as np, pandas as pd, scanpy as sc, matplotlib.pyplot as plt
from scipy.stats import rankdata
import torch

print('GPU available:', torch.cuda.is_available())
OUT = '/kaggle/working' if os.path.exists('/kaggle/working') else 'outputs'
os.makedirs(OUT, exist_ok=True)
paths = glob.glob('../Dataset/**/E9.5_E1S1*.h5ad', recursive=True) + glob.glob('**/E9.5_E1S1*.h5ad', recursive=True)
if not paths:
    raise FileNotFoundError('Add E9.5_E1S1.MOSTA.h5ad as an input dataset (Add Input -> Upload)')
adata = sc.read_h5ad(paths[0]); adata.var_names_make_unique()
ANN = 'annotation'
adata.obs[ANN] = adata.obs[ANN].astype('category')

GPU available: True


In [4]:
def rank01(t):
    t = np.asarray(t, dtype=float)
    t = np.where(np.isfinite(t), t, np.nanmax(t[np.isfinite(t)]))
    return (rankdata(t) - 1) / (len(t) - 1)

LINEAGE = ['Neural crest', 'Branchial arch', 'Mesenchyme']
ROOT_TYPE = 'Neural crest'
ad = adata[adata.obs[ANN].isin(LINEAGE)].copy()
ad.obs[ANN] = ad.obs[ANN].cat.remove_unused_categories()
if 'count' in ad.layers:
    ad.X = ad.layers['count'].copy()
sc.pp.filter_genes(ad, min_cells=10)
raw = ad.copy()                      # raw counts, kept for SpaceFlow (it normalises internally)

# same preprocessing + DPT root as the PyCharm notebook
sc.pp.normalize_total(ad, target_sum=1e4); sc.pp.log1p(ad)
sc.pp.highly_variable_genes(ad, n_top_genes=2000)
sc.tl.pca(ad, n_comps=30, mask_var='highly_variable')
sc.pp.neighbors(ad, n_neighbors=15, n_pcs=30)
sc.tl.diffmap(ad, n_comps=15)
dc1 = ad.obsm['X_diffmap'][:, 1]
idx = np.where(ad.obs[ANN] == ROOT_TYPE)[0]
root = idx[np.argmin(dc1[idx])] if np.median(dc1[idx]) < np.median(dc1) else idx[np.argmax(dc1[idx])]
ad.uns['iroot'] = int(root); sc.tl.dpt(ad)
ad.obs['t_dpt_pca'] = rank01(ad.obs['dpt_pseudotime'])
xy = np.asarray(ad.obsm['spatial'])[:, :2]
print(ad.shape, '| root spot:', ad.obs_names[root])

/home/user/Gagan_env/genomicsProject/.genomicsProjectENV/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(1545, 16491) | root spot: 230_108


## 1. Method 1 — DPT (aspatial)

In [5]:
def save_pt(col, name):
    pd.DataFrame({'cell_id': ad.obs_names, 'x': xy[:, 0], 'y': xy[:, 1], 'tissue': ad.obs[ANN].values,
                  'pseudotime': ad.obs[col].values, 'method': name}).to_csv(f'{OUT}/pt_{name}.csv', index=False)
    print(name, '|', ad.obs.groupby(ANN, observed=True)[col].mean().sort_values().round(3).to_dict())

save_pt('t_dpt_pca', 'dpt_pca')

dpt_pca | {'Neural crest': 0.353, 'Branchial arch': 0.59, 'Mesenchyme': 0.885}


## 2. Method 2 — PAGA (aspatial, cluster graph)

In [ ]:
from scipy.sparse.csgraph import shortest_path
sc.tl.leiden(ad, resolution=1.0, key_added='clusters', flavor='igraph', n_iterations=2, directed=False)
sc.tl.paga(ad, groups='clusters')
C = ad.uns['paga']['connectivities'].toarray()
cats = list(ad.obs['clusters'].cat.categories)
r = cats.index(ad.obs['clusters'].iloc[int(root)])
W = np.where(C > 0.05, 1.0 / np.maximum(C, 1e-9), 0)
dist = shortest_path(W, directed=False, indices=r)
dist[~np.isfinite(dist)] = np.nanmax(dist[np.isfinite(dist)]) + 1
Xp = ad.obsm['X_pca']; d_root = np.linalg.norm(Xp - Xp[int(root)], axis=1)
gap = np.diff(np.unique(dist)).min() if len(np.unique(dist)) > 1 else 1.0
t = ad.obs['clusters'].map(pd.Series(dist, index=cats)).astype(float).values + 0.5 * gap * d_root / d_root.max()
ad.obs['t_paga'] = rank01(t)
save_pt('t_paga', 'paga')

## 3. Method 3 — SpaceFlow (spatial: graph neural network, then pseudotime on its embedding)

In [6]:
from SpaceFlow import SpaceFlow
import networkx as nx, scipy.sparse as sp
# compatibility fix: SpaceFlow 1.0.4 calls a networkx function that newer networkx removed
if not hasattr(nx, 'to_scipy_sparse_matrix'):
    nx.to_scipy_sparse_matrix = lambda G, format='csr', **kw: sp.csr_matrix(nx.to_scipy_sparse_array(G, format=format, **kw))

EPOCHS = 1000          # SpaceFlow default; early stopping usually ends sooner
sf = SpaceFlow.SpaceFlow(adata=raw.copy())
sf.preprocessing_data(n_top_genes=2000)          # normalise, log, HVGs, PCA, spatial graph (alpha complex)
emb = sf.train(embedding_save_filepath=f'{OUT}/spaceflow_embedding.tsv',
               spatial_regularization_strength=0.1, z_dim=50, lr=1e-3,
               epochs=EPOCHS, max_patience=50, min_stop=100, random_seed=42, gpu=0)
print('embedding shape:', emb.shape)

/home/user/Gagan_env/genomicsProject/.genomicsProjectENV/lib/python3.10/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Epoch 2/1000, Loss: 2.047470808029175
Epoch 12/1000, Loss: 1.4499101638793945
Epoch 22/1000, Loss: 1.3992235660552979
Epoch 32/1000, Loss: 1.292029619216919
Epoch 42/1000, Loss: 1.0202505588531494
Epoch 52/1000, Loss: 0.6194320321083069
Epoch 62/1000, Loss: 0.33000093698501587
Epoch 72/1000, Loss: 0.2565782964229584
Epoch 82/1000, Loss: 0.14888238906860352
Epoch 92/1000, Loss: 0.14814873039722443
Epoch 102/1000, Loss: 0.12750405073165894
Epoch 112/1000, Loss: 0.0992436558008194
Epoch 122/1000, Loss: 0.09527091681957245
Epoch 132/1000, Loss: 0.07234569638967514
Epoch 142/1000, Loss: 0.08468468487262726
Epoch 152/1000, Loss: 0.07043597847223282
Epoch 162/1000, Loss: 0.0692719891667366
Epoch 172/1000, Loss: 0.06238333880901337
Epoch 182/1000, Loss: 0.06962654739618301
Epoch 192/1000, Loss: 0.08371493220329285
Epoch 202/1000, Loss: 0.07155710458755493
Epoch 212/1000, Loss: 0.05794607475399971
Epoch 222/1000, Loss: 0.06773141026496887
Epoch 232/1000, Loss: 0.04771369323134422
Epoch 242/1000

In [7]:
e = sc.AnnData(emb); e.obs_names = ad.obs_names
sc.pp.neighbors(e, n_neighbors=20, use_rep='X')
sc.tl.diffmap(e, n_comps=15)
e.uns['iroot'] = int(root)
sc.tl.dpt(e)
ad.obs['t_spaceflow'] = rank01(e.obs['dpt_pseudotime'].values)
save_pt('t_spaceflow', 'spaceflow')

spaceflow | {'Neural crest': 0.433, 'Mesenchyme': 0.531, 'Branchial arch': 0.788}


## 4. Method 4 — SpaTrack (spatial: optimal transport)
**Idea:** the cost of a cell "moving" to another cell mixes **expression distance** and **physical distance**.
Entropic optimal transport (Sinkhorn) turns these costs into a cell-to-cell **transition probability matrix**.
Starting from the spots around our neural crest root, cells that receive the most transition probability are earliest;
cells that receive the least are latest. `ALPHA_SPACE` and `ALPHA_EXPR` set how much each kind of distance counts (paper default: 1 and 1).

In [10]:
import numpy as np
if not hasattr(np, 'Inf'):
    np.Inf = np.inf          # compatibility fix: spaTrack 0.1.1 uses a name removed in NumPy 2
import spaTrack as spt

ALPHA_SPACE, ALPHA_EXPR = 1, 1
st = ad.copy()
st.obsm['X_spatial'] = xy
st.obs['cluster'] = ad.obs[ANN].astype(str)
st.obsp['trans'] = spt.get_ot_matrix(st, data_type='spatial', alpha1=ALPHA_SPACE, alpha2=ALPHA_EXPR)
start_cells = spt.set_start_cells(st, select_way='coordinates', start_point=tuple(xy[root]),
                                  basis='spatial', n_neigh=20)
print('start cells:', len(start_cells), '| their tissues:', st.obs[ANN].iloc[list(start_cells)].value_counts().to_dict())
ad.obs['t_spatrack'] = rank01(spt.get_ptime(st, list(start_cells)))
save_pt('t_spatrack', 'spatrack')

start cells: 20 | their tissues: {'Neural crest': 20, 'Branchial arch': 0, 'Mesenchyme': 0}
spatrack | {'Neural crest': 0.369, 'Branchial arch': 0.504, 'Mesenchyme': 0.891}


/home/user/Gagan_env/genomicsProject/.genomicsProjectENV/lib/python3.10/site-packages/spaTrack/single_time/velocity.py:204: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  ptime[cell_tran_sort[i]] = i / (adata.n_obs - 1)


## 5. Summary — all methods side by side

In [11]:
cols = ['t_dpt_pca', 't_paga', 't_spaceflow', 't_spatrack']
names = ['DPT', 'PAGA', 'SpaceFlow', 'SpaTrack']
means = ad.obs.groupby(ANN, observed=True)[cols].mean().round(3)
means.columns = names
print('Mean pseudotime per tissue (0 = early):'); print(means.loc[['Neural crest', 'Branchial arch', 'Mesenchyme']])
A = ad.obs[cols].corr(method='spearman').round(3); A.index = A.columns = names
print('\nAgreement between methods (Spearman):'); print(A)

fig, axes = plt.subplots(1, 5, figsize=(26, 5.5))
for lab in ad.obs[ANN].cat.categories:
    m = (ad.obs[ANN] == lab).values
    axes[0].scatter(xy[m, 0], xy[m, 1], s=5, label=lab)
axes[0].legend(fontsize=8); axes[0].set_title('Lineage tissues')
for a, c, n in zip(axes[1:], cols, names):
    s = a.scatter(xy[:, 0], xy[:, 1], c=ad.obs[c], s=5, cmap='viridis')
    a.scatter(xy[root, 0], xy[root, 1], c='red', s=100, marker='*'); a.set_title(n)
    plt.colorbar(s, ax=a, fraction=0.046)
for a in axes: a.set_aspect('equal')
plt.tight_layout(); plt.show()
print(sorted(glob.glob(f'{OUT}/pt_*.csv')))

KeyError: "Columns not found: 't_paga'"

## 6. Download the results
Run this cell and click each blue link (or use the Output panel). Better: **Save Version → Save & Run All (Commit)** to keep them permanently.

In [12]:
from IPython.display import FileLink, display
os.chdir(OUT)
for f in sorted(glob.glob('pt_*.csv')):
    display(FileLink(f))

/media/user/EE16-375E/Course Work PhD/Computational Genomics/Project/Codes/outputs/pt_dpt_pca.csv

/media/user/EE16-375E/Course Work PhD/Computational Genomics/Project/Codes/outputs/pt_spaceflow.csv

/media/user/EE16-375E/Course Work PhD/Computational Genomics/Project/Codes/outputs/pt_spatrack.csv